In [ ]:
import sys
import os
from pathlib import Path

# Detect project root and cd into it (important for relative paths like results/logs)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

# Add /src to PYTHONPATH so we can import llm_abm_ga without installing the package
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project ROOT:", ROOT)
print("Using CWD:", Path.cwd())
print("SRC exists:", SRC.exists())

In [ ]:
import llm_abm_ga.config as config

# 1. Dynamically configure the LLM Provider and Model
config.LLM_PROVIDER = "openai"       # Choose "openai" or "gemini"
config.LLM_MODEL = "gpt-4o-mini"     # e.g., "gpt-4o-mini", "gemini-2.5-flash"
config.LLM_TEMPERATURE = 0.0         # Deterministic outputs for stability

In [ ]:
from llm_abm_ga.prompts.components import system_template_phrases, user_template_phrases
from llm_abm_ga.prompts.templates import decode_individual

# 2. Define your optimized prompt configuration using 0s and 1s.
# The baseline system prompt has exactly 8 components in components.py.
# Replace this array with the "best individual" found by your GA!
system_genome = [1, 0, 0, 1, 1, 1, 0, 0] 

# The user prompt has 1 component (usually left empty, so we use 0)
user_genome = [0] 

# Decode the genome. This automatically translates the 1s into phrases 
# AND appends the MANDATORY_SYSTEM_LINE and MANDATORY_USER_LINE.
individual = (system_genome, user_genome)
llm_config = decode_individual(
    individual=individual,
    system_phrases=system_template_phrases,
    user_phrases=user_template_phrases
)

# Let's print it to verify the mandatory lines are there
print("--- DECODED SYSTEM PROMPT ---")
print(llm_config["system_template"])
print("\n--- DECODED USER PROMPT ---")
print(llm_config["user_template"])

# Wrap in a list as MarketModel expects a list of configs
firm_prompt_configs = [llm_config]

In [ ]:
from llm_abm_ga.market.model import MarketModel

print(f"\nInitializing hybrid market with {config.LLM_PROVIDER.upper()} ({config.LLM_MODEL})...")

# 3. Initialize the market: 1 LLM firm vs 10 Rule-Based firms
model = MarketModel(
    firm_prompt_configs=firm_prompt_configs,
    n_rule_firms=10,         # 10 rule-based baseline firms
    n_buyers=1000,         # 100k buyers to minimize stochastic noise
    periods=10               # 10 trading periods
)

# 4. Run the simulation
print("Starting market simulation...")
model.run()
print("Simulation complete!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# 1. Extract the final cumulative profits from the model
llm_profit = next(firm.cumulative_profit for firm in model.firms if firm.id == 0)
rule_based_profits = [firm.cumulative_profit for firm in model.firms if firm.id != 0]

# 2. Calculate the average and standard deviation for the rule-based firms
avg_rule_based_profit = np.mean(rule_based_profits)
std_rule_based_profit = np.std(rule_based_profits)

# 3. Setup data for the plot
labels = ["Optimized LLM", "Rule-Based Avg"]
means = [llm_profit, avg_rule_based_profit]
errs = [0, std_rule_based_profit] # 0 standard deviation for the single LLM firm

# ---- IEEE-like style ----
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times"],
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "figure.dpi": 300,
})

# ---- Colors ----
colors = ["#1fd4b0", "#fcbb43"]

# ---- Plot ----
fig, ax = plt.subplots(figsize=(3.4, 2.4))

bars = ax.bar(
    labels,
    means,
    yerr=errs,
    capsize=4,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
    zorder=3
)

ax.set_ylabel("Profit", fontsize=10)
ax.axhline(0, color="black", linewidth=0.8, zorder=1)

# ---- Y padding for a clean look ----
m, e = np.array(means), np.array(errs)
upper, lower = m + e, m - e
pad = 0.08 * (upper.max() - lower.min())
ax.set_ylim(lower.min() - pad, upper.max() + pad)

# ---- Ticks ----
ax.tick_params(axis="x", pad=4)
ax.tick_params(axis="y", labelsize=9)

fig.tight_layout()
plt.show()

# Print text summary
print("\n--- Final Market Standings (Summary) ---")
print(f"LLM Firm: {llm_profit:.2f} total profit")
print(f"Rule-Based Firms (Average): {avg_rule_based_profit:.2f} ± {std_rule_based_profit:.2f} total profit")